In [45]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import ClassifierChain

import pickle
import time 
import pandas as pd

## Load and Preprocess Data

In [46]:
# Load the data
df = pd.read_csv("./datasets/bemo_prompts_dataset_v2.1.csv")

In [47]:
# Show 5 random rows
df.sample(5)

,Unnamed: 0,prompt,general,smart_home,todo,mail,learning_resources,others
1340,973,Draft a thank-you email to my professor.,0,0,0,1,0,0
1714,1347,Update me on generative AI developments. Book...,1,0,1,1,1,0
606,239,Please add 'buy groceries' to my task list.,0,0,1,0,0,0
1508,1141,"I need to know the capital of France, and I al...",1,1,1,1,0,0
769,402,I'd appreciate it if you could explain reinfor...,1,0,1,1,1,0


In [48]:
# Drop the first column
df.drop(df.columns[0], axis=1, inplace=True)

In [49]:
df.sample(5)

,prompt,general,smart_home,todo,mail,learning_resources,others
1695,Please remind me to submit the report; I'd als...,1,0,1,0,0,0
1436,Add to my to-do list: Investigate the learning...,1,1,1,0,1,0
2118,Set a reminder to submit the report and send t...,0,0,1,1,0,0
2274,Email me a GitHub repository showcasing GAN ex...,1,0,1,1,1,0
20,What is your opinion on art?,0,0,0,0,0,1


In [50]:
# Split the data into X and Y
x = df["prompt"]
Y = df.drop(columns=["prompt"])
labels = Y.columns

In [51]:
print(x.shape, Y.shape)

(3056,) (3056, 6)


In [52]:
# Train-test split
x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size=0.3, random_state=42, shuffle=True)

### Vanilla Classifier Chain with Logistic Regression

In [53]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(LogisticRegression(), verbose=True)

In [54]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [55]:
# Train the model
start = time.time()
pipeline.fit(x_train, Y_train)
end = time.time()

print(f"Training time: {end-start}")

[Chain] ................... (1 of 6) Processing order 0, total=   0.0s
[Chain] ................... (2 of 6) Processing order 1, total=   0.1s
[Chain] ................... (3 of 6) Processing order 2, total=   0.1s


[Chain] ................... (4 of 6) Processing order 3, total=   0.1s
[Chain] ................... (5 of 6) Processing order 4, total=   0.0s
[Chain] ................... (6 of 6) Processing order 5, total=   0.0s
Training time: 0.36667442321777344


In [56]:
# Predict
Y_pred = pipeline.predict(x_test)

In [57]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.7960741548527808
Hamming Loss: 0.045438022537259176
F1 Score (micro): 0.929932735426009


In [58]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = pipeline.predict(new_prompt)
end = time.time()

In [59]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("Input:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)

Input: i hate you
Predicted Tasks: ['others']
Time taken: 0.006885051727294922


In [60]:
# Save the model
with open("./models/TC_Pipeline_LR_v3.pkl", "wb") as f:
    pickle.dump(pipeline, f)

In [61]:
# Save the labels
with open("./models/labels_v2.pkl", "wb") as f:
    pickle.dump(labels, f)

In [62]:
# Test the saved model
with open("./models/TC_Pipeline_LR_v3.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)
    
with open("./models/labels_v2.pkl", "rb") as f:
    labels = pickle.load(f)
    
new_prediction = loaded_pipeline.predict(new_prompt)
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("Input:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)

Input: i hate you
Predicted Tasks: ['others']


### Vanilla Classifier Chain with XGBoost

In [22]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(GradientBoostingClassifier(), verbose=True)

In [23]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [24]:
# Train the model
start = time.time()
pipeline.fit(x_train, Y_train)
end = time.time()
print(f"Training time: {end-start}")

[Chain] ................... (1 of 6) Processing order 0, total=   0.9s
[Chain] ................... (2 of 6) Processing order 1, total=   0.9s
[Chain] ................... (3 of 6) Processing order 2, total=   0.8s
[Chain] ................... (4 of 6) Processing order 3, total=   0.8s
[Chain] ................... (5 of 6) Processing order 4, total=   0.8s
[Chain] ................... (6 of 6) Processing order 5, total=   0.8s
Training time: 4.862981557846069


In [25]:
# Predict
Y_pred = pipeline.predict(x_test)

In [26]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.7786259541984732
Hamming Loss: 0.042711741185023626
F1 Score (micro): 0.9342657342657342


In [29]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = pipeline.predict(new_prompt)
end = time.time()

In [30]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("Input:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)

Input: show me my tasks
Predicted Tasks: ['todo']
Time taken: 0.00899648666381836


In [31]:
# Save the model
with open("./models/TC_Pipeline_GB_v2.pkl", "wb") as f:
    pickle.dump(pipeline, f)

### Classifier Chain with Logistic Regression and Grid Search for Hyperparameter Tuning

In [32]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(LogisticRegression())

In [33]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [34]:
# Define hyperparameter grid
param_grid = {
    'classifier__base_estimator__C': [0.01, 0.1, 1, 10],  # Regularization strength
    'classifier__base_estimator__penalty': ['l1', 'l2', 'elasticnet'],  # Regularization type
    'classifier__base_estimator__solver': ['liblinear', 'saga'],  # Solver (some only support specific penalties)
    'classifier__base_estimator__max_iter': [100, 200, 500],  # Number of iterations
    'classifier__base_estimator__tol': [1e-4, 1e-3, 1e-2],  # Stopping criteria tolerance
    'classifier__base_estimator__class_weight': [None, 'balanced'],  # Handle imbalanced data
    'vectorizer__max_features': [100, 500, 1000],  # Number of TF-IDF features
    'vectorizer__ngram_range': [(1, 1), (1, 2)]  # Use unigrams or bigrams
}

In [35]:
# Initialize the grid search
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)

In [36]:
# Train the model
start = time.time()
grid_search.fit(x_train, Y_train)
end = time.time()
print(f"Training time: {end-start}")

Fitting 3 folds for each of 2592 candidates, totalling 7776 fits
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l1, classifier__base_estimator__solver=liblinear, classifier__base_estimator__tol=0.0001, vectorizer__max_features=500, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l1, classifier__base_estimator__solver=liblinear, classifier__base_estimator__tol=0.0001, vectorizer__max_features=500, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l1, classifier__base_estimator__solver=liblinear, classifier__base_estimator__tol=0

/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.0001, vectorizer__max_features=100, vectorizer__ngram_range=(1, 2); total time=   0.4s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.0001, vectorizer__max_features=500, vectorizer__ngram_range=(1, 1); total time=   0.3s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.0001, vectorizer__max_features=500, vectorizer__ngram_range=(1, 1); total time=

/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.01, vectorizer__max_features=100, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.001, vectorizer__max_features=1000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END classifier__base_estimator__C=0.01, classifier__base_estimator__class_weight=None, classifier__base_estimator__max_iter=100, classifier__base_estimator__penalty=l2, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.01, vectorizer__max_features=100, vectorizer__ngram_range=(1, 1); total time=   0

...

[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.001, vectorizer__max_features=100, vectorizer__ngram_range=(1, 2); total time=   0.1s
[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.001, vectorizer__max_features=500, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.001, vectorizer__max_features=100, vectorizer__ngra

/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
2592 fits failed out of a total of 7776.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1296 fits failed with the following error:
Traceback (most recent call last):
  File "/home/momad/projects/BEMO/.venv/lib/python3.10/site-pack

[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.01, vectorizer__max_features=100, vectorizer__ngram_range=(1, 2); total time=   0.1s
[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.01, vectorizer__max_features=1000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END classifier__base_estimator__C=10, classifier__base_estimator__class_weight=balanced, classifier__base_estimator__max_iter=500, classifier__base_estimator__penalty=elasticnet, classifier__base_estimator__solver=saga, classifier__base_estimator__tol=0.01, vectorizer__max_features=1000, vectorizer__ngram

/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [ 0.  0.  0. ... nan nan nan]
  warnings.warn(


Training time: 228.31357049942017


In [37]:
# Print best parameters and best score
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Best Parameters: {'classifier__base_estimator__C': 10, 'classifier__base_estimator__class_weight': 'balanced', 'classifier__base_estimator__max_iter': 100, 'classifier__base_estimator__penalty': 'l1', 'classifier__base_estimator__solver': 'saga', 'classifier__base_estimator__tol': 0.01, 'vectorizer__max_features': 1000, 'vectorizer__ngram_range': (1, 1)}
Best Score: 0.8910705937353903


In [38]:
# Get the best model
best_pipeline = grid_search.best_estimator_

In [39]:
# Predict
Y_pred = best_pipeline.predict(x_test)

In [40]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.9083969465648855
Hamming Loss: 0.018538713195201745
F1 Score (micro): 0.9723127035830619


In [41]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = best_pipeline.predict(new_prompt)
end = time.time()

In [42]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("Input:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)

Input: show me resources about java and add it to my tasks
Predicted Tasks: ['general', 'todo', 'learning_resources']
Time taken: 0.007146596908569336


In [44]:
# Save the best model
with open("./models/TC_Pipeline_LR_v4.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)